# JGI Integration Workflow

This notebook runs the unified multi-omics integration pipeline.  
Select the analysis track in `config_examples/analysis.yml → analysis.track`:

| Track | When to use | Integration strategy |
|---|---|---|
| `sample_resolution` | Replicates **are** paired across data types (same biological material, matching sample IDs) | log2 → align by Sample ID → row-wise Z-score per dataset → concatenate → Pearson/Spearman correlation → HDBSCAN / network clustering |
| `condition_resolution` | Replicates are **not** paired | log2 → collapse to condition means → Centroid or LFC sub-approach → row-wise Z-score → concatenate → correlation → hierarchical / HDBSCAN / network clustering |

**Pipeline (both tracks):**
```
linked_data
  → filter_data              (remove rare features)
  → devariance_data          (remove low-variance features)
  → replicate_filtering      (remove outlier replicates)
  ↓
  [sample_resolution only]
  → scale_all_datasets       (log2 + row-wise Z-score per dataset)
  ↓
  → integrate_data           (concatenate; condition_resolution also collapses + Z-scores here)
  → perform_feature_selection (variance or feature_list filter)
  → group_features           (network_modules | hierarchical_clustering | hdbscan)
```

# Project setup

In [ ]:
import os
import sys
import yaml
import pandas as pd
import tools.objects as objs
import tools.helpers as hlp
pd.options.display.max_columns = None
pd.set_option('display.max_colwidth', 500)

In [ ]:
# Optional: list all configurations available to load with the hash arguments below
hlp.list_persistent_configs()

In [ ]:
# Initialize project (add the data processing and analysis hashes if resuming a previous run)
project = objs.Project(data_processing_hash=None, analysis_hash=None, overwrite=False)

# Dataset initiation

In [ ]:
# Create dataset objects and gather raw metadata and data (already pre-obtained from JGI sources)
tx_dataset = objs.TX(project)
mx_dataset = objs.MX(project, last=True)

# Analysis initiation

In [ ]:
# Create analysis object — track is automatically detected from analysis.yml → analysis.track
analysis = objs.Analysis(project, datasets=[tx_dataset, mx_dataset])
print(f"Track: {analysis.integration_mode}")

# Link Samples Between Datasets

In [ ]:
# Link analysis datasets by finding corresponding sample metadata fields
analysis.link_metadata()

In [ ]:
# Link analysis dataset matrices using linked metadata
analysis.link_data()

# Data processing

Steps shared by both tracks. Parameters are set in `data_processing.yml`.

In [ ]:
# Filter out rare features from analysis datasets based on minimum observed value
analysis.filter_all_datasets()

In [ ]:
# Filter out features from analysis datasets that were not impacted by experimentation
analysis.devariance_all_datasets()

In [ ]:
# Filter out features with outlier replicates from each dataset
analysis.replicability_test_all_datasets()

In [ ]:
# View raw data distributions before any scaling
hlp.plot_simple_histogram(mx_dataset.replicate_filtered_data, plot_title="Metabolite Peak Heights (raw)", output_dir=analysis.output_dir)
hlp.plot_simple_histogram(tx_dataset.replicate_filtered_data, plot_title="Transcript Counts (raw)", output_dir=analysis.output_dir)

# Per-dataset scaling (sample_resolution track only)

In [ ]:
# Scale each dataset independently (log2 + row-wise Z-score). Automatically skipped in condition_resolution track.
analysis.scale_all_datasets()

# Integration analysis

In [ ]:
# Integrate metadata tables across datasets
analysis.integrate_metadata()

In [ ]:
# Integrate data matrices according to the selected track
analysis.integrate_data()

In [ ]:
# Annotate the integrated features with pre-generated feature annotation tables
analysis.annotate_integrated_features()

In [ ]:
# Subset features by variance or a user-supplied feature list (set in analysis.yml → feature_selection)
analysis.perform_feature_selection()

In [ ]:
# View PCA of the integrated matrix to examine clustering pattern of combined data types
hlp.plot_simple_pca(
    analysis.integrated_data_selected,
    analysis.integrated_metadata,
    title=f"PCA of {analysis.integration_mode} integrated data",
    output_dir=analysis.output_dir
)

# Find data-driven groups of features

Set `analysis.feature_grouping.method` in `analysis.yml`:  
- `network_modules` — Pearson/Spearman correlation graph + Louvain/Leiden community detection  
- `hierarchical_clustering` — scipy hierarchical clustering (1 − r distance)  
- `hdbscan` — density-based clustering

In [ ]:
# Group features by co-variation pattern across samples/conditions
analysis.group_features()

# Data-driven vs. knowledge-driven feature grouping

In [ ]:
# Compare data-driven feature groups against biological pathway annotations
grouping_results = analysis.compare_groups_to_pathways()

# Exploration of Results

In [ ]:
# View average abundance of a given group across samples/conditions
analysis.plot_submodule_avg_abundance(submodule_name='group_1', metadata_cat='group', save_plot=True)

In [ ]:
# View abundance patterns of individual features
analysis.plot_individual_feature(feature_id='mx_11781_positive', metadata_cat='group', save_plot=True)

In [ ]:
# View pairwise feature correlation (sample_resolution track)
hlp.plot_feature_pair_correlation(
    data=analysis.integrated_data_selected,
    metadata=analysis.integrated_metadata,
    feature_1="tx_gene_12834",
    feature_2="mx_11781_positive",
    color_by="timepoint",
    output_dir=analysis.output_dir
)

# Save configuration and notebook

In [ ]:
# Save persistent configuration and notebook files
project.save_persistent_config_and_notebook()